In [1]:
from ift6758.notebooks.llm_tests import profile_name
%load_ext autoreload
%autoreload 2

In [2]:
from ift6758.llm.prompts import PROFILES,OG_TACHE_1
from ift6758.llm.rag import load_doc, extract_section, split_doc, load_embedder, embed, retrieve, format_context,DOCS_DIR
from ift6758.llm.qwen_log import create_log,log_call
from ift6758.llm.qwen_utils import load_qwen,build_messages,generate,ask_qwen
from ift6758.llm.test_both_code import check_download, check_cache_no_request, check_failed_request,check_wrong_cache, run_tests
from ift6758.llm.get_better import extract_text,build_refine_prompt,load_function,format_test_report
import requests
import json
from pathlib import Path
from tempfile import TemporaryDirectory

In [3]:
filename = "nhl-api-reference.md"
url = "https://raw.githubusercontent.com/Zmalski/NHL-API-Reference/main/README.md"

text = load_doc(url, filename)
section = extract_section(text,"# NHL Web API Documentation","# NHL Stats API Documentation")
chunks = split_doc(section)
embedder = load_embedder()
vectors = embed(chunks,embedder)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
GAME_ID = 2022030411
raw_path = Path(DOCS_DIR.parent / "raw") / f"{GAME_ID}.json"
print("Cached game:", raw_path.resolve())
if not raw_path.exists():
    response = requests.get(f"https://api-web.nhle.com/v1/gamecenter/{GAME_ID}/play-by-play", timeout=30)
    response.raise_for_status()
    downloaded = response.json()
    assert downloaded["id"] == GAME_ID and isinstance(downloaded["plays"], list)
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    raw_path.write_text(response.text, encoding="utf-8")
game = json.loads(raw_path.read_text(encoding="utf-8")) # Convertie de json vers objet python
assert game["id"] == GAME_ID and isinstance(game["plays"], list)
print(game["awayTeam"]["abbrev"], "at", game["homeTeam"]["abbrev"], game["gameDate"])

Cached game: /Users/jcforget/Maitrise/Cours/IFT 6758/Projet/Github/NHL-data-analysis/ift6758/data/raw/2022030411.json
FLA at VGK 2023-06-03


In [4]:
model, tokenizer, device = load_qwen()
log_path = create_log()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [ ]:
task,iteration,MAX_ITER = OG_TACHE_1, 0 ,5
profile_name = "gen_code"
try:
    while True:
        profile_name = "gen_code"
        p = PROFILES[profile_name]
        #confi rag.py
        query = p["retrieval_query"] or task
        results = retrieve(query,chunks,vectors,embedder,k=p["k"])
        context = format_context(results,source=filename)
        print("=== Itération", iteration, "— génération du code ===")
        gen_result = ask_qwen(model,tokenizer,task,context,p,log_call(),log_path)